# RUKOPYS Qwen3-VL Stage 2C: Formula/Table Only Fine-tune

This notebook fine-tunes the current Qwen3-VL LoRA adapter only on the `formula` and `table` crops from the no-padding augmented dataset.

Expected input dataset layout:

```text
ocr_region_crops_wrong_aug/
  labels.jsonl
  formula/*.jpg
  table/*.jpg
  stats.json
```

The loader filters `labels.jsonl` to `formula` and `table` only. It uses `text` as the answer and rebuilds the same source-aware crop OCR prompt used by the hybrid pipeline.


In [ ]:
# Kaggle dependency cell.
INSTALL_DEPS = True

if INSTALL_DEPS:
    import subprocess
    import sys

    commands = [
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            "accelerate",
            "peft",
            "bitsandbytes",
            "trl",
            "qwen-vl-utils",
            "datasets",
            "pandas==2.2.2",
            "pillow<12",
        ],
        [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/transformers.git"],
    ]
    for cmd in commands:
        print("Running:", " ".join(cmd), flush=True)
        subprocess.check_call(cmd)


In [ ]:
import gc
import json
import os
import random
import time
from collections import Counter
from pathlib import Path

import torch
from datasets import Dataset

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1"
START_LORA_DIR = "/kaggle/input/datasets/trankimhuu/qwen3vl-rukopys-lora-final"

# Set this to the Kaggle input folder that contains labels.jsonl plus formula/table folders.
OCR_CROPS_ROOT = "/kaggle/input/datasets/trankimhuu/ocr-region-crops-wrong-aug/ocr_region_crops_wrong_aug"
OCR_CROPS_ROOT_CANDIDATES = [
    OCR_CROPS_ROOT,
    "/kaggle/input/ocr-region-crops-wrong-aug/ocr_region_crops_wrong_aug",
    "/kaggle/input/ocr-region-crops-wrong-aug",
    "/kaggle/working/ocr_region_crops_wrong_aug",
    "artifacts/ocr_region_crops_wrong_aug",
]

OUTPUT_DIR = Path("/kaggle/working/qwen3vl_rukopys_stage2c_formula_table_aug")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_TYPES = {"formula", "table"}
MAX_TRAIN_SAMPLES = None  # Optional smoke limit; keep None for the full formula/table dataset.

NUM_TRAIN_EPOCHS = 3
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8
DATALOADER_NUM_WORKERS = 2
MODEL_DEVICE_MAP = "balanced"
MAX_SEQ_LENGTH = 3072
MAX_PIXELS_CROP = 320_000

PRINT_STEPS = 10
SAVE_STEPS = 500
SAVE_TOTAL_LIMIT = 4
RESUME_TRAINING = True
RESUME_CHECKPOINT_DIR = ""  # Optional: set to a mounted Kaggle input checkpoint dir.


In [ ]:
SOURCE_HINTS = {
    "dictation": "Ukrainian dictation handwriting. Do not complete from canonical text; read only visible characters.",
    "archive": "Historical Ukrainian/Cyrillic document. Preserve old spelling; do not modernize.",
    "school": "School homework. It may contain corrections, teacher marks, formulas, and mixed handwriting/print.",
    "university": "University exam/coursework. It may contain formulas, tables, chemistry notation, and technical symbols.",
}
DEFAULT_SOURCE_HINT = "Read only visible characters from this crop."

SPECIAL_TEXT_MARKER_RULES = (
    "Use [illegible] only for unreadable words inside an otherwise legible text region. "
    "Use ~~word~~ for visible strikethrough and ~~old~~{new} for visible correction."
)

STAGE_B_GUARDRAILS = (
    "The final transcription must be supported by the crop. "
    "Do not complete missing words from source hint, language prior, or canonical dictation text. "
    "Do not translate, correct grammar, normalize spelling, expand abbreviations, summarize, "
    "or infer hidden/missing text. No JSON, no Markdown, no explanation."
)

CROP_PROMPTS = {
    "formula": (
        "Read this standalone math, logic, vector, matrix, determinant, set/relation, statistics, physics, "
        "or chemistry expression exactly as written. Return only formula text, using LaTeX when it is the "
        "clearest representation and plain Unicode when it better matches the handwriting. Preserve visible "
        "symbols, indices, superscripts, subscripts, arrows, fractions, matrix/determinant structure, punctuation, "
        "numbering, and strikethrough/correction markers. Do not solve, simplify, normalize, explain, or convert "
        "old notation into a different style."
    ),
    "table": (
        "Read this table region exactly. Return only pipe-separated table text. Use one output line per visual row "
        "and | between cells. Preserve empty cells with empty fields, for example A||C. Preserve row order, "
        "column order, multi-word cell text, wrapped cell text, numbers, units, punctuation, dashes, and visible "
        "spelling mistakes. Do not infer missing cells, do not rebalance columns, do not summarize, and do not explain."
    ),
}


def normalize_source(value):
    value = str(value or "").strip().lower()
    return value if value in SOURCE_HINTS else "default"


def build_crop_prompt(region_type, source=None):
    source_hint = SOURCE_HINTS.get(normalize_source(source), DEFAULT_SOURCE_HINT)
    type_prompt = CROP_PROMPTS[region_type]
    return "\n".join([source_hint, type_prompt, SPECIAL_TEXT_MARKER_RULES, STAGE_B_GUARDRAILS])


def find_model_id():
    item = str(BASE_MODEL_PATH)
    if item.startswith("/") and Path(item).exists():
        return item
    if not item.startswith("/"):
        return item
    raise FileNotFoundError(f"No Qwen3-VL model found at BASE_MODEL_PATH={item}")


def find_start_lora_dir():
    p = Path(START_LORA_DIR)
    if (p / "adapter_config.json").exists():
        return p
    raise FileNotFoundError(f"No LoRA adapter_config.json found at START_LORA_DIR={p}")


def find_ocr_crops_root():
    for item in OCR_CROPS_ROOT_CANDIDATES:
        p = Path(item)
        if (p / "labels.jsonl").exists():
            return p
    raise FileNotFoundError(
        "No labels.jsonl found. Update OCR_CROPS_ROOT to the folder containing ocr_region_crops_wrong_aug."
    )


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_cached_image_path(data_root, row):
    raw = Path(str(row.get("crop_path") or row.get("image_path") or row.get("relative_image_path") or ""))
    if raw.is_absolute() and raw.exists():
        return str(raw)
    candidates = [data_root.parent / raw, data_root / raw, data_root / raw.name]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    raise FileNotFoundError(f"Cached crop not found for row crop_path={raw}")


def label_to_sample(data_root, row):
    region_type = str(row.get("type") or "").strip().lower()
    if region_type not in TARGET_TYPES:
        return None
    answer = "" if row.get("text") is None else str(row.get("text"))
    if not answer.strip():
        return None
    return {
        "image_path": resolve_cached_image_path(data_root, row),
        "prompt": build_crop_prompt(region_type, source=row.get("source")),
        "answer": answer,
        "region_type": region_type,
        "source": row.get("source", "unknown"),
        "augment": bool(row.get("augment")),
        "aug_id": int(row.get("aug_id") or 0),
        "source_kind": row.get("source_kind", "unknown"),
        "crop_path": row.get("crop_path", ""),
    }


model_id = find_model_id()
start_lora_dir = find_start_lora_dir()
ocr_crops_root = find_ocr_crops_root()
labels_path = ocr_crops_root / "labels.jsonl"
stats_path = ocr_crops_root / "stats.json"

labels = read_jsonl(labels_path)
samples = [label_to_sample(ocr_crops_root, row) for row in labels]
samples = [row for row in samples if row is not None]
random.Random(SEED).shuffle(samples)
if MAX_TRAIN_SAMPLES is not None:
    samples = samples[:MAX_TRAIN_SAMPLES]

if not samples:
    raise RuntimeError(f"No formula/table training samples found in {labels_path}")

stage2_ds = Dataset.from_list(samples)
prompt_config = {
    "stage": "stage2c_formula_table_aug_dataset",
    "target_types": sorted(TARGET_TYPES),
    "source_hints": SOURCE_HINTS,
    "default_source_hint": DEFAULT_SOURCE_HINT,
    "crop_prompts": CROP_PROMPTS,
    "special_text_marker_rules": SPECIAL_TEXT_MARKER_RULES,
    "stage_b_guardrails": STAGE_B_GUARDRAILS,
    "crop_prompt_assembly": "source_hint + type_prompt + special_text_marker_rules + stage_b_guardrails",
}
if stats_path.exists():
    prompt_config["data_stats"] = json.loads(stats_path.read_text(encoding="utf-8"))

print("Base model:", model_id)
print("Starting LoRA:", start_lora_dir)
print("OCR crops root:", ocr_crops_root)
print("Training samples:", len(samples))
print("Counts by type:", dict(Counter(row["region_type"] for row in samples)))
print("Augmented by type:", dict(Counter(row["region_type"] for row in samples if row.get("augment"))))
print("Source kind counts:", dict(Counter(row.get("source_kind", "unknown") for row in samples)))


In [ ]:

from peft import PeftModel, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig, TrainerCallback
from trl import SFTConfig, SFTTrainer


def force_fp16_config(model):
    model.config.torch_dtype = torch.float16
    for attr in ("text_config", "vision_config"):
        cfg = getattr(model.config, attr, None)
        if cfg is not None:
            cfg.torch_dtype = torch.float16
            if hasattr(cfg, "dtype"):
                cfg.dtype = "float16"


processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

max_memory = {0: "14GiB"}
if torch.cuda.device_count() > 1:
    max_memory[1] = "14GiB"

base_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map=MODEL_DEVICE_MAP,
    max_memory=max_memory,
    quantization_config=quantization_config,
    dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
force_fp16_config(base_model)
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
try:
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    base_model.gradient_checkpointing_enable()

model = PeftModel.from_pretrained(base_model, str(start_lora_dir), is_trainable=True)
for _, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
model.print_trainable_parameters()


In [ ]:

def build_messages(sample):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image_path"], "max_pixels": MAX_PIXELS_CROP},
                {"type": "text", "text": sample["prompt"]},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": sample["answer"]}]},
    ]


def encode_marker(tokenizer):
    try:
        return tokenizer.encode("<|im_start|>assistant\n", allowed_special="all", add_special_tokens=False)
    except TypeError:
        return tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)


ASSISTANT_MARKER = encode_marker(processor.tokenizer)


def data_collator(examples):
    messages_list = [build_messages(ex) for ex in examples]
    texts = [
        processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in messages_list
    ]
    image_inputs, video_inputs = process_vision_info(messages_list)
    batch = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100

    eos_id = processor.tokenizer.eos_token_id
    for i in range(labels.shape[0]):
        ids = batch["input_ids"][i].tolist()
        start = -1
        for j in range(0, len(ids) - len(ASSISTANT_MARKER) + 1):
            if ids[j:j + len(ASSISTANT_MARKER)] == ASSISTANT_MARKER:
                start = j + len(ASSISTANT_MARKER)
                break
        actual_len = int(batch["attention_mask"][i].sum().item())
        truncated = actual_len >= MAX_SEQ_LENGTH and (eos_id is None or ids[actual_len - 1] != eos_id)
        if start >= 0 and not truncated:
            labels[i, :start] = -100
        else:
            labels[i, :] = -100

    batch["labels"] = labels
    for key, value in list(batch.items()):
        if isinstance(value, torch.Tensor) and value.dtype == torch.float32:
            batch[key] = value.to(torch.float16)
    return batch


In [ ]:

class PrintProgressCallback(TrainerCallback):
    def __init__(self, name):
        self.name = name
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"[{self.name}] start: max_steps={state.max_steps}, grad_accum={args.gradient_accumulation_steps}", flush=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        elapsed = time.time() - (self.start_time or time.time())
        step = max(1, state.global_step)
        eta = (elapsed / step) * max(0, state.max_steps - step)
        loss = logs.get("loss", logs.get("eval_loss", None))
        lr = logs.get("learning_rate", None)
        msg = f"[{self.name}] step {state.global_step}/{state.max_steps}"
        if loss is not None:
            msg += f" loss={loss:.4f}"
        if lr is not None:
            msg += f" lr={lr:.2e}"
        msg += f" elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m"
        if torch.cuda.is_available():
            msg += " vram=" + ",".join(
                f"{i}:{torch.cuda.memory_allocated(i)/1024**3:.1f}GB"
                for i in range(torch.cuda.device_count())
            )
        print(msg, flush=True)

    def on_step_end(self, args, state, control, **kwargs):
        step = int(state.global_step or 0)
        if step <= 0 or step % PRINT_STEPS != 0:
            return
        elapsed = time.time() - (self.start_time or time.time())
        eta = (elapsed / max(1, step)) * max(0, state.max_steps - step)
        msg = f"[{self.name}] print step {step}/{state.max_steps} elapsed={elapsed/60:.1f}m eta={eta/60:.1f}m"
        if torch.cuda.is_available():
            msg += " vram=" + ",".join(
                f"{i}:{torch.cuda.memory_allocated(i)/1024**3:.1f}GB"
                for i in range(torch.cuda.device_count())
            )
        print(msg, flush=True)


class OOMRecoverySFTTrainer(SFTTrainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        try:
            try:
                loss = super().training_step(model, inputs, num_items_in_batch=num_items_in_batch)
            except TypeError:
                loss = super().training_step(model, inputs)
            if self.args.device != loss.device:
                loss = loss.to(self.args.device)
            return loss
        except torch.cuda.OutOfMemoryError:
            print("OOM: skipping one batch after clearing cache.", flush=True)
            for p in model.parameters():
                p.grad = None
            torch.cuda.empty_cache()
            gc.collect()
            return torch.tensor(0.0, device=self.args.device)


def find_latest_checkpoint(root):
    root = Path(root)
    if not root.exists():
        return None
    checkpoints = []
    for path in root.glob("checkpoint-*"):
        try:
            step = int(path.name.rsplit("-", 1)[-1])
        except ValueError:
            continue
        if (path / "trainer_state.json").exists():
            checkpoints.append((step, path))
    if not checkpoints:
        return None
    return str(max(checkpoints, key=lambda item: item[0])[1])


def resolve_resume_checkpoint(output_dir):
    if not RESUME_TRAINING:
        return None
    if RESUME_CHECKPOINT_DIR:
        resume_root = Path(RESUME_CHECKPOINT_DIR)
        if not resume_root.exists():
            raise FileNotFoundError(f"RESUME_CHECKPOINT_DIR does not exist: {resume_root}")
        if (resume_root / "trainer_state.json").exists():
            return str(resume_root)
        latest = find_latest_checkpoint(resume_root)
        if latest:
            return latest
        raise FileNotFoundError(f"No checkpoint-* with trainer_state.json found under {resume_root}")
    return find_latest_checkpoint(output_dir)


training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR / "stage2c_formula_table_aug"),
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=8e-6,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_pin_memory=False,
    dataloader_persistent_workers=DATALOADER_NUM_WORKERS > 0,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = OOMRecoverySFTTrainer(
    model=model,
    args=training_args,
    train_dataset=stage2_ds,
    data_collator=data_collator,
    callbacks=[PrintProgressCallback("stage2c-formula-table-aug")],
)
resume_checkpoint = resolve_resume_checkpoint(training_args.output_dir)
if resume_checkpoint:
    print("Resuming from checkpoint:", resume_checkpoint, flush=True)
else:
    print("No resume checkpoint found; starting a fresh Stage 2C formula/table run.", flush=True)
trainer.train(resume_from_checkpoint=resume_checkpoint)

final_dir = OUTPUT_DIR / "qwen3vl_rukopys_stage2c_formula_table_aug_lora_final"
trainer.model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)

prompt_config.update({
    "stage": "stage2c_formula_table_aug_cached_finetune",
    "learning_rate": 8e-6,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "save_steps": SAVE_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "resume_from_checkpoint": resume_checkpoint,
    "base_model": str(model_id),
    "start_lora": str(start_lora_dir),
    "ocr_crops_root": str(ocr_crops_root),
    "labels_path": str(labels_path),
})
(final_dir / "rukopys_prompt_config.json").write_text(
    json.dumps(prompt_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved final Stage 2C formula/table LoRA adapter to", final_dir)


In [ ]:
# Optional quick sanity check on one formula/table crop. This is not a leaderboard estimate.
RUN_QUICK_SANITY = True

if RUN_QUICK_SANITY and samples:
    model.eval()
    sample = next((row for row in samples if row.get("region_type") == "formula"), samples[0])
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image_path"], "max_pixels": MAX_PIXELS_CROP},
                {"type": "text", "text": sample["prompt"]},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16):
        out = model.generate(**inputs, max_new_tokens=192, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out)]
    pred = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    print("Prediction:", pred[:1000])
    print("Target:", sample.get("answer", "")[:1000])
